# CALCOLO PARAMETRI SARIMA

In [ ]:
# =====================================================================
# SCRIPT AUTOMATICO PARAMETRI SARIMA
# =====================================================================
# Esegui questo comando per installare la libreria (se non è già installata)
!pip install pmdarima

import pandas as pd
import numpy as np
import os
import pmdarima as pm
from datetime import timedelta
from google.colab import drive
import warnings

# Ignora i warning statistici per mantenere la console pulita
warnings.filterwarnings('ignore')

# ---------------------------------------------------------------------
# 1. SETUP E CONNESSIONE DRIVE
# ---------------------------------------------------------------------
drive.mount('/content/drive')

# Modifica il percorso se necessario
PATH = '/content/drive/MyDrive/Colab Notebooks/TIROCINIO/dataset_ricostruito/'
FILE_NAME = 'traffico_r_completo.csv'

# ---------------------------------------------------------------------
# 2. FUNZIONI DI CARICAMENTO E PRE-PROCESSING
# ---------------------------------------------------------------------
def carica_dataset_completo(nome_file):
    print(f"Lettura del dataset {nome_file} in corso...")
    percorso_completo = os.path.join(PATH, nome_file)
    if not os.path.exists(percorso_completo):
        raise FileNotFoundError(f"File non trovato nel percorso: {percorso_completo}")

    df = pd.read_csv(percorso_completo)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    return df

def pre_processing_stazione(df_completo, id_stazione):
    # Filtraggio
    df_stazione = df_completo[df_completo['id_stazione'] == id_stazione].copy()
    df_stazione.set_index('timestamp', inplace=True)
    df_stazione.sort_index(inplace=True)

    # Aggregazione a 30 Minuti
    df_resampled = df_stazione[['differenza']].resample('30min').mean()
    df_resampled['differenza'] = df_resampled['differenza'].ffill()

    # Feature Temporali
    df_resampled['hour'] = df_resampled.index.hour
    df_resampled['dayofweek'] = df_resampled.index.dayofweek
    df_resampled['is_weekend'] = (df_resampled.index.dayofweek >= 5).astype(int)

    df_resampled.dropna(inplace=True)
    return df_resampled

# ---------------------------------------------------------------------
# 3. ESECUZIONE RICERCA PARAMETRI (AUTO-ARIMA)
# ---------------------------------------------------------------------
if __name__ == "__main__":

    # Parametri per evitare il Data Leakage (Non guardiamo i dati di test!)
    GIORNI_DI_TEST_FUTURI = 14

    # Caricamento file pesante
    try:
        dataset_raw = carica_dataset_completo(FILE_NAME)
    except FileNotFoundError as e:
        print(e)
        exit()

    stazioni_disponibili = dataset_raw['id_stazione'].unique()
    dizionario_parametri_finali = {}

    print("\n" + "="*70)
    print(" INIZIO RICERCA PARAMETRI OTTIMALI SARIMA (Stagionalità = 48 slot)")
    print(f" Stazioni da elaborare: {len(stazioni_disponibili)} -> {list(stazioni_disponibili)}")
    print(" ATTENZIONE: Questa operazione richiederà tempo. Lasciare il PC acceso.")
    print("="*70 + "\n")

    for st_id in stazioni_disponibili:
        print(f"--> Analisi Stazione ID: {st_id} in corso...")

        # 1. Estrazione dati puliti per la stazione
        df_stazione = pre_processing_stazione(dataset_raw, st_id)

        # 2. Taglio temporale: escludiamo dal tuning i giorni che userai per il test
        data_fine_tuning = df_stazione.index.max() - timedelta(days=GIORNI_DI_TEST_FUTURI)
        df_train_tuning = df_stazione[df_stazione.index < data_fine_tuning]

        endog = df_train_tuning['differenza'].values

        print(f"    [Dati estratti: {len(endog)} slot storici. Ricerca in corso...]")

        # 3. Ricerca Automatica
        # Nota: Limitiamo la ricerca (max_p, max_q) per evitare crash di RAM su Colab
        try:
            auto_model = pm.auto_arima(
                endog,
                seasonal=True, m=48,          # Stagionalità a 48 slot (24 ore)
                max_p=2, max_q=2,             # Limite parametri base
                max_P=1, max_Q=1,             # Limite parametri stagionali
                trace=False,                  # Non stampiamo ogni singolo tentativo
                error_action='ignore',
                suppress_warnings=True,
                stepwise=True,                # Usa algoritmo intelligente per fare prima
                n_jobs=-1                     # Usa tutti i core disponibili
            )

            # 4. Salvataggio
            dizionario_parametri_finali[st_id] = {
                'order': auto_model.order,
                'seasonal_order': auto_model.seasonal_order
            }
            print(f"    [OK] Trovati: Order={auto_model.order}, Seasonal={auto_model.seasonal_order}\n")

        except Exception as e:
            print(f"    [ERRORE] Impossibile calcolare per la stazione {st_id}: {e}\n")

    # ---------------------------------------------------------------------
    # 4. OUTPUT FINALE DA COPIARE
    # ---------------------------------------------------------------------
    print("\n\n" + "#"*70)
    print(" RICERCA CONCLUSA CON SUCCESSO! ")
    print(" Copia l'intero dizionario qui sotto e incollalo nel tuo Notebook Principale:")
    print("#"*70 + "\n")

    print("PARAMETRI_SARIMA_STAZIONI = {")
    for st_id, params in dizionario_parametri_finali.items():
        print(f"    {st_id}: {params},")
    print("}")
    print("\n" + "#"*70)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 7.7 MB/s eta 0:00:00
Mounted at /content/drive
Lettura del dataset traffico_r.csv in corso...

 INIZIO RICERCA PARAMETRI OTTIMALI SARIMA (Stagionalità = 48 slot)
 Stazioni da elaborare: 9 -> [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]
 ATTENZIONE: Questa operazione richiederà tempo. Lasciare il PC acceso.

--> Analisi Stazione ID: 1 in corso...
    [Dati estratti: 10947 slot storici. Ricerca in corso...]
